In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             1544795
   Errors:                    678
   Search Artists:            1803604
   Known Summary IDs:         1455842


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [29]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  223384
#   Artist Names To Get:  201834
#   Artist Names To Get:  182959
#   Artist Names To Get:  149609

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   1677695
#   Artist Names To Get:  125909


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "9:50am")
tt = TermTime("today", "10:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-07-03 10:14:54
========================= termTime(day=today,time=10:00pm) =========================
   ====> Terminate Time Set To 2022-07-03 22:00:00 <====
   ====> Will Terminate Process 11 Hours and 45 Minutes From Now
Getting Pra Elas (0004189896)                             True
Getting R. E. O'Brien (0000325002)                        True
Getting R. E. Pope (0000386291)                           True
Getting R. E. Wetherald (0000702556)                      True
Getting R. E. Dias (0001048857)                           True
Getting R. E. Berryhill (0003090183)                      True
Getting E.  Leusner / R. Chamfleury (0002248889)          True
Getting Chang-Yun Yo (0002443854)                         True
Getting E. Claudette Freeman (0001485788)                 True
Getting M. Freeman (0002956060)                           True
Getting Ypey (0000593776)                                 True
Getting E. Lacrouis (0

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Christianus de Hollande (0003330508)              True
Getting Yoyito Cabrera (0000593567)                       True
Getting Yoyito Cabrera (0001878720)                       True
Getting Cabrera Normando (0003174475)                     True
Getting Cabrera HD (0003621515)                           True
Getting D. Cabrera (0002204557)                           True
Getting Ángel Cabrera (0003488517)                        True
Getting Cabrera (0001784076)                              True
Getting Cabrera (0003896584)                              True
Getting Cabrera (0003965054)                              True
Getting Yousefi (0001407315)                              True
Getting Tuki Witika (0004145216)                          True
Getting Tuki (0002395691)                                 True
Getting Ewan (0002518235)                                 True
Getting Ewan (0003280925)                                 True
Getting 3rei D (0002515068)                            

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Pauline Darkling (0003843650)                     True
Getting Dreiklang Berlin Ensemble (0000582595)            True
Getting Dasi Bom Project (0003436225)                     True
Getting 3k (0004162075)                                   True
Getting 3k Short (0001683832)                             True
Getting The 3K Band (0002638358)                          True
Getting Every 3k (0003821387)                             True
Getting Yu Jhu (0004216810)                               True
Getting Yuji (0001301915)                                 True
Getting Tassajara Symphony (0001698626)                   True
Getting Stephanie Desjour (0001556701)                    True
Getting Martina Dieziger (0002374184)                     True
Getting Sebastien Desjours (0003856165)                   True
Getting Nils Desjours (0004047650)                        True
Getting Yuji Ando (0002588881)                            True
Getting 3hrs (0003075637)                              

...

Getting Valarie Grace (0001433418)                        True
Getting Grace Day (0001829568)                            True
Getting Grace Tea (0001528282)                            True
Getting TT Grace (0002515237)                             True
Getting Christy Day (0001768551)                          True
Getting Daniel Toscan du Plantier (0002197043)            True
Getting Daniel Toscan Duplantier (0001607055)             True
Getting Grace Daily (0002637072)                          True
Getting Della Grace (0002746807)                          True
Getting Rik Wolton (0001418419)                           True
Getting Che Wolton Grant (0003577771)                     True
Getting Amy Whilden (0001780651)                          True
Getting Amy Walton (0003800891)                           True
Getting Aimee Walden (0004208161)                         True
Getting Tracey Lamb (0001041547)                          True
Getting Grace Cooke (0001938156)                       

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Daniel Tejedor (0002749148)                       True
Getting Beaulieu Porch (0003092151)                       True
Getting Trace Element (0000897386)                        True
Getting Franzen & Trace Dog (0001840631)                  True
Getting Trace Element (0001835825)                        True
Getting Trace Element (0003332058)                        True
Getting Trace Elements (0003433605)                       True
Getting Daniel Thornton (0003561600)                      True
Getting Donal Thornton (0003405855)                       True
Getting Grace McKenna (0002602097)                        True
Getting Rolf Meyn (0001415916)                            True
Getting Butch Meyn (0001555700)                           True
Getting Marie Caruso (0002048435)                         True
Getting Marie Garza (0003406356)                          True
Getting Marie Anne Cayrouze (0002055022)                  True
Getting Jimmy Lee Boggs (0003298260)                   

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Daniel Ferrandis (0003887441)                     True
Getting Tragedia (0003614182)                             True
Getting Lady Traffic (0002015828)                         True
Getting Bok-Sung Ryu & the Traffic Lights (0003895434)    True
Getting Daniel Zielske (0000957322)                       True
Getting Allen Pringle (0003066376)                        True
Getting Xay Trafficant'e (0000465531)                     True
Getting Santo Trafficante (0001573612)                    True
Getting Kirk Trafficante (0003307957)                     True
Getting Valerie Traficante (0002346090)                   True
Getting Debra Traficante (0003677082)                     

In [ ]:
from lib.allmusic import moveLocalFiles
moveLocalFiles()

In [ ]:
tmp = []
for line in tmp:
    if line.startswith("Getting"):
        artistID = line.split()[-2][1:-1]
        print(artistID)
        del searchedForErrors[artistID]

# Create Tabs Data

In [ ]:
def getTabs(rData):
    extraData = rData.profile.extra
    tabs = extraData.get('tabs', []) if isinstance(extraData,dict) else []
    retval = {tab.title: tab.href for tab in tabs}
    return retval

In [ ]:
mio   = allmusic.MusicDBIO()
tabsData = None
for modVal in range(100):
    modValTabsData = mio.data.getModValData(modVal).apply(getTabs)
    tabsData = concat([tabsData, modValTabsData]) if isinstance(tabsData,Series) else modValTabsData

# Download Artist Discography Data

# Parse

In [ ]:
from utils import PoolIO
pio = PoolIO("AllMusic")
pio.parse(force=True)
pio.meta()
pio.sum()
pio.search()